In [ ]:
# Install required packages (runs via micropip in JupyterLite, pip locally)
%pip install pandas matplotlib numpy ipywidgets

# Browser History Topic Overview

Extracted from `browser_history_domain_overview.ipynb` starting at original Cell 21.\n

In [ ]:
from initialisation import make_upload_widget

# Renders an "Upload CSV" button with a live status area below it.
# Pick your CSV file, wait for the ✅ message, then run the next cell.
# (If running locally with a *.csv already present, skip this cell.)
uploader = make_upload_widget()

In [ ]:
import pandas as pd
from initialisation import load_history_df

df = load_history_df(uploader=uploader)
print(f"Loaded {len(df):,} rows for topic analysis")

## 12) Expanded Topic Classification – Phase 1
Classifies **all** visited URLs (not just Google searches) using a priority-based approach:
1. **Domain mapping** (`DOMAIN_CATEGORY_MAP`) — fastest, most reliable
2. **Page title keywords** — contextual clues
3. **Domain name keywords** — fallback for unmapped domains
4. **URL path keywords** — last resort

Bilingual keywords (English + German). First match wins.\n

In [ ]:
from initialisation import DOMAIN_CATEGORY_MAP, TOPICS

print(f"DOMAIN_CATEGORY_MAP: {len(DOMAIN_CATEGORY_MAP)} domains")
print(f"TOPICS: {len(TOPICS)} categories, {sum(len(v) for v in TOPICS.values())} total keywords")

In [ ]:
from initialisation import classify_df

classify_df(df)
print(f"Classified {len(df):,} total visits")

In [ ]:
# Category distribution across ALL visits
category_counts = df["category"].value_counts()
total = len(df)

print(f"Category distribution – all {total:,} visits\n")
print(f"{'Category':<25} {'Count':>8}  {'%':>6}")
print("-" * 45)
for cat, count in category_counts.items():
    pct = count / total * 100
    print(f"{cat:<25} {count:>8,}  {pct:>5.1f}%")

other_pct = category_counts.get("Other", 0) / total * 100
# Threshold from Phase 1 plan: investigate clustering if > 40% falls into "Other"
OTHER_THRESHOLD_PCT = 40
print(f"\n→ 'Other' coverage: {other_pct:.1f}% ",
      "(Phase 2 clustering recommended)" if other_pct > OTHER_THRESHOLD_PCT else "(coverage acceptable)")

# Top 5 domains per non-Other category
print("\n\nTop 5 domains per category (excluding 'Other'):\n")
for cat in category_counts.index:
    if cat == "Other":
        continue
    top_domains = df[df["category"] == cat]["domain"].value_counts().head(5)
    domain_str = ", ".join(f"{d}({c})" for d, c in top_domains.items())
    print(f"{cat:<25} {domain_str}")

## 13) Phase 2 – Unsupervised Clustering of 'Other' Visits
Uses **TF-IDF char n-grams** + **KMeans** to discover natural groupings inside the
unclassified 'Other' bucket, helping identify topics that the keyword lists missed.\n

In [ ]:
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")

other_df = df[df["category"] == "Other"].copy()
print(f"'Other' visits: {len(other_df):,} ({len(other_df) / len(df) * 100:.1f}% of total)")

# Minimum "Other" entries needed to make clustering statistically meaningful
MIN_CLUSTER_SIZE = 30
if len(other_df) < MIN_CLUSTER_SIZE:
    print("Not enough 'Other' entries to cluster meaningfully — skipping.")
else:
    page_title_col = "PageTitle" if "PageTitle" in other_df.columns else None
    title_series = (
        other_df[page_title_col].fillna("") if page_title_col else pd.Series("", index=other_df.index)
    )
    other_df["_cluster_text"] = other_df["domain"].fillna("") + " " + title_series

    # MAX_CLUSTERS caps granularity; MIN_CLUSTERS ensures meaningful groups;
    # CLUSTER_SIZE_RATIO: aim for ~50 visits per cluster
    MAX_CLUSTERS = 20
    MIN_CLUSTERS = 5
    CLUSTER_SIZE_RATIO = 50
    n_clusters = min(MAX_CLUSTERS, max(MIN_CLUSTERS, len(other_df) // CLUSTER_SIZE_RATIO))

    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 3), max_features=500)
    X = vectorizer.fit_transform(other_df["_cluster_text"] )

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    other_df["cluster"] = kmeans.fit_predict(X)

    print(f"\nDiscovered {n_clusters} clusters in 'Other' visits.")
    print("Review these to find new keyword categories:\n")
    print(f"{'Cluster':>8}  {'Size':>6}  Top domains (count)")
    print("-" * 70)
    for cid in range(n_clusters):
        rows = other_df[other_df["cluster"] == cid]
        top = rows["domain"].value_counts().head(4)
        top_str = ', '.join(f"{d}({c})" for d, c in top.items())
        print(f"{cid:>8}  {len(rows):>6}  {top_str}")

    # Expose cluster labels on the main df for further inspection
    # Use boolean-mask positional assignment to avoid duplicate-index reindex errors
    other_mask = df["category"] == "Other"
    df.loc[other_mask, "other_cluster"] = other_df["cluster"].astype("Int64").to_numpy()
    print("\nCluster IDs written to df['other_cluster'] for further inspection.")

In [ ]:
df['other_cluster']